<a href="https://colab.research.google.com/github/flahbocchino/cardioia-fase2-diagnostico/blob/main/cardioia_parte2_classificador_risco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CardioIA - Fase 2
## Parte 2: Classificador de Risco com TF-IDF e Machine Learning

**Aluna:** Flavia Nunes Bocchino
**RM:** 564213
**Curso:** Inteligencia Artificial - FIAP

---

### O que este notebook faz?

Este codigo simula um sistema de triagem clinica automatizada.
Ele analisa frases com sintomas e classifica o nivel de risco do paciente.

Etapas:
1. Carrega um dataset de frases medicas ja rotuladas (alto risco / baixo risco)
2. Aplica TF-IDF para transformar texto em numeros que o computador entende
3. Treina um modelo de Machine Learning para aprender o padrao
4. Testa o modelo com novas frases e avalia sua acuracia

### O que e TF-IDF?
TF-IDF (Term Frequency - Inverse Document Frequency) e uma tecnica
que transforma palavras em numeros, dando mais peso para palavras
que sao importantes e unicas em um texto. Assim o algoritmo consegue
"ler" e comparar frases.

> **Importante:** Este sistema e academico e nao substitui triagem medica profissional.

In [1]:
# CELULA 1 - Importacao das bibliotecas
# sklearn: biblioteca de Machine Learning mais usada em Python
# TfidfVectorizer: transforma texto em numeros usando TF-IDF
# DecisionTreeClassifier: modelo de arvore de decisao
# LogisticRegression: modelo de regressao logistica
# train_test_split: divide os dados em treino e teste
# accuracy_score, classification_report: medem o desempenho do modelo

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


In [2]:
# CELULA 2 - Criacao do dataset de frases rotuladas
# Cada frase representa o relato de um paciente
# O rotulo indica se e alto risco ou baixo risco
# Quanto mais frases, melhor o modelo aprende

dados = {
    "frase": [
        "sinto dor no peito e falta de ar ha dois dias",
        "tive um leve incomodo nas costas ontem",
        "meu coracao dispara e sinto tontura frequente",
        "estou com dor de cabeca leve e cansaco",
        "sinto pressao forte no peito irradiando para o braco esquerdo",
        "tive uma dor muscular apos exercicio fisico",
        "acordei com falta de ar e suor frio durante a noite",
        "sinto um leve formigamento nas maos as vezes",
        "dor no peito com nausea e suor frio ha uma hora",
        "estou com tosse seca e dor de garganta leve",
        "meu coracao bate muito rapido e irregular ha tres dias",
        "sinto cansaco apos caminhadas longas mas passa rapido",
        "pressao alta com dor de cabeca forte e visao embacada",
        "tive um enjoo leve apos comer algo pesado",
        "formigamento no braco esquerdo com dor no peito constante",
        "dor leve nas costas que melhora com repouso",
        "palpitacoes fortes com sensacao de desmaio e fraqueza",
        "sinto um leve cansaco no final do dia de trabalho",
        "falta de ar intensa em repouso com pernas inchadas",
        "tive uma dor de cabeca que passou com agua",
        "dor no peito que piora com respiracao e febre alta",
        "sinto um leve desconforto no estomago apos refeicao",
        "perdi a consciencia por alguns segundos e acordei confuso",
        "estou com coriza e espirros desde ontem",
        "suor frio intenso com dor irradiando para o pescoco",
        "dor muscular leve apos academia ontem",
        "ritmo cardiaco irregular com tontura e falta de ar",
        "sinto um pouco de ansiedade antes de reunioes",
        "coracao acelerado com pressao alta e dor no torax",
        "leve dor de cabeca que passa com repouso",
        "inchazo nas pernas e dificuldade para respirar ao deitar",
        "sinto cansaco leve no fim de semana apos atividades",
        "dor forte no peito com suor frio e mal estar geral",
        "tive um arroto e leve desconforto no peito apos comer",
        "pressao muito alta com confusao mental e dificuldade de falar",
        "dor nas costas leve que melhora com alongamento",
        "sensacao de aperto no peito com falta de ar ao subir escadas",
        "leve tontura ao levantar rapido da cama pela manha",
        "palpitacoes com desmaio e extremidades frias e palidas",
        "sinto um cansaco normal apos dia longo de trabalho"
    ],
    "situacao": [
        "alto risco", "baixo risco", "alto risco", "baixo risco",
        "alto risco", "baixo risco", "alto risco", "baixo risco",
        "alto risco", "baixo risco", "alto risco", "baixo risco",
        "alto risco", "baixo risco", "alto risco", "baixo risco",
        "alto risco", "baixo risco", "alto risco", "baixo risco",
        "alto risco", "baixo risco", "alto risco", "baixo risco",
        "alto risco", "baixo risco", "alto risco", "baixo risco",
        "alto risco", "baixo risco", "alto risco", "baixo risco",
        "alto risco", "baixo risco", "alto risco", "baixo risco",
        "alto risco", "baixo risco", "alto risco", "baixo risco"
    ]
}

df = pd.DataFrame(dados)

df.to_csv("frases_risco.csv", index=False)

print("Dataset criado com " + str(len(df)) + " frases!")
print("Distribuicao de classes:")
print(df['situacao'].value_counts().to_string())

Dataset criado com 40 frases!
Distribuicao de classes:
situacao
alto risco     20
baixo risco    20


In [3]:
# CELULA 3 - Aplicar TF-IDF para transformar texto em numeros
# O TF-IDF converte cada frase em um vetor numerico
# Isso permite que o algoritmo de ML processe o texto

df = pd.read_csv("frases_risco.csv")

# Separa as frases (X) dos rotulos (y)
X = df['frase']
y = df['situacao']

# Divide em 80% para treino e 20% para teste
# random_state=42 garante que a divisao seja sempre igual
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Total de frases: " + str(len(df)))
print("Frases para treino: " + str(len(X_treino)))
print("Frases para teste: " + str(len(X_teste)))
print()

# Aplica o TF-IDF
# max_features=100: usa as 100 palavras mais relevantes
tfidf = TfidfVectorizer(max_features=100)

# fit_transform: aprende o vocabulario e transforma o treino
X_treino_tfidf = tfidf.fit_transform(X_treino)

# transform: usa o vocabulario ja aprendido para transformar o teste
X_teste_tfidf = tfidf.transform(X_teste)

print("TF-IDF aplicado com sucesso!")
print("Cada frase agora e representada por " + str(X_treino_tfidf.shape[1]) + " numeros")

Total de frases: 40
Frases para treino: 32
Frases para teste: 8

TF-IDF aplicado com sucesso!
Cada frase agora e representada por 100 numeros


In [4]:
# CELULA 4 - Treinar e avaliar os modelos
# Vamos testar dois modelos e comparar qual tem melhor acuracia

print("=" * 55)
print("   TREINAMENTO E AVALIACAO DOS MODELOS")
print("=" * 55)

# ------ MODELO 1: Arvore de Decisao ------
# Funciona como um fluxograma de perguntas e respostas
print("\nMODELO 1: Arvore de Decisao")
modelo_arvore = DecisionTreeClassifier(random_state=42)
modelo_arvore.fit(X_treino_tfidf, y_treino)
pred_arvore = modelo_arvore.predict(X_teste_tfidf)
acuracia_arvore = accuracy_score(y_teste, pred_arvore)
print("Acuracia: " + str(round(acuracia_arvore * 100, 2)) + "%")

# ------ MODELO 2: Regressao Logistica ------
# Calcula a probabilidade de cada classe
print("\nMODELO 2: Regressao Logistica")
modelo_logistic = LogisticRegression(random_state=42, max_iter=1000)
modelo_logistic.fit(X_treino_tfidf, y_treino)
pred_logistic = modelo_logistic.predict(X_teste_tfidf)
acuracia_logistic = accuracy_score(y_teste, pred_logistic)
print("Acuracia: " + str(round(acuracia_logistic * 100, 2)) + "%")

# ------ Escolha o melhor modelo ------
print("\n" + "=" * 55)
if acuracia_logistic >= acuracia_arvore:
    melhor_modelo = modelo_logistic
    melhor_pred = pred_logistic
    melhor_acuracia = acuracia_logistic
    nome_modelo = "Regressao Logistica"
else:
    melhor_modelo = modelo_arvore
    melhor_pred = pred_arvore
    melhor_acuracia = acuracia_arvore
    nome_modelo = "Arvore de Decisao"

print("Melhor modelo: " + nome_modelo)
print("Acuracia final: " + str(round(melhor_acuracia * 100, 2)) + "%")
print("=" * 55)
print()
print("Relatorio completo do melhor modelo:")
print(classification_report(y_teste, melhor_pred))

   TREINAMENTO E AVALIACAO DOS MODELOS

MODELO 1: Arvore de Decisao
Acuracia: 75.0%

MODELO 2: Regressao Logistica
Acuracia: 100.0%

Melhor modelo: Regressao Logistica
Acuracia final: 100.0%

Relatorio completo do melhor modelo:
              precision    recall  f1-score   support

  alto risco       1.00      1.00      1.00         4
 baixo risco       1.00      1.00      1.00         4

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8



In [5]:
# CELULA 5 - Testar com novas frases nunca vistas pelo modelo
# Esta etapa simula o uso real do sistema em triagem clinica

print("=" * 55)
print("   TESTE COM NOVAS FRASES")
print("=" * 55)

novas_frases = [
    "sinto dor forte no peito e nao consigo respirar direito",
    "tive uma dor de cabeca leve que ja passou",
    "coracao acelerado com tontura e suor frio intenso",
    "sinto um leve cansaco depois de trabalhar muito",
    "dor no peito irradiando para o braco com nausea",
    "estou com dor muscular leve apos caminhada",
    "perdi os sentidos por alguns segundos e acordei confuso",
    "sinto um leve desconforto estomacal apos o almoco"
]

novas_tfidf = tfidf.transform(novas_frases)
predicoes = melhor_modelo.predict(novas_tfidf)

for i, (frase, pred) in enumerate(zip(novas_frases, predicoes), 1):
    if pred == "alto risco":
        nivel = "ALTO RISCO - Atendimento imediato recomendado"
    else:
        nivel = "BAIXO RISCO - Monitoramento recomendado"
    print("\nFrase " + str(i) + ": " + frase)
    print("Classificacao: " + nivel)
    print("-" * 55)

print("\nClassificacao concluida!")
print("Lembrete: Sistema academico. Sempre consulte um medico.")

   TESTE COM NOVAS FRASES

Frase 1: sinto dor forte no peito e nao consigo respirar direito
Classificacao: ALTO RISCO - Atendimento imediato recomendado
-------------------------------------------------------

Frase 2: tive uma dor de cabeca leve que ja passou
Classificacao: BAIXO RISCO - Monitoramento recomendado
-------------------------------------------------------

Frase 3: coracao acelerado com tontura e suor frio intenso
Classificacao: ALTO RISCO - Atendimento imediato recomendado
-------------------------------------------------------

Frase 4: sinto um leve cansaco depois de trabalhar muito
Classificacao: BAIXO RISCO - Monitoramento recomendado
-------------------------------------------------------

Frase 5: dor no peito irradiando para o braco com nausea
Classificacao: ALTO RISCO - Atendimento imediato recomendado
-------------------------------------------------------

Frase 6: estou com dor muscular leve apos caminhada
Classificacao: BAIXO RISCO - Monitoramento recomendado